
# Autonomous Multi-Agent Startup Studio with Modal

## Install packages

In [ ]:
!pip install -q --upgrade \
    modal \
    openai==1.12.0 \
    sentence-transformers==2.7.0 \
    chromadb==0.4.24 \
    gradio==4.19.0 \
    python-dotenv==1.0.0 \
    pydantic==2.10.6

## Setup

In [18]:

import os
import json
import logging
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from google.colab import userdata
import chromadb
from sentence_transformers import SentenceTransformer
import modal

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["MODAL_TOKEN_ID"] = userdata.get("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = userdata.get("MODAL_TOKEN_SECRET")

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
openai_client = OpenAI()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("startup_studio")
logger.setLevel(logging.INFO)

## Write the Modal service file

In [3]:

startup_modal_service = """
import os
import modal
from openai import OpenAI

app = modal.App("startup-studio-service")

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("openai>=1.12.0")
)

secrets = [modal.Secret.from_name("openai-secret")]

def _client():
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


@app.cls(image=image, secrets=secrets, timeout=600)
class IdeaGenerator:
    @modal.method()
    def generate(self, theme: str, memory_context: str = "", n_ideas: int = 5) -> str:
        prompt = f\"\"\"
You are the IdeaGeneratorAgent in an autonomous startup studio.

Generate {n_ideas} realistic startup ideas for this theme:
{theme}

Use these memory examples for inspiration only, and do not copy them:
{memory_context if memory_context else "No related ideas in memory."}

Return valid JSON with this shape:
{{
  "ideas": [
    {{
      "name": "...",
      "one_liner": "...",
      "problem": "...",
      "solution": "...",
      "target_market": "...",
      "business_model": "...",
      "moat": "...",
      "founder_fit": "..."
    }}
  ]
}}
\"\"\"

        response = _client().chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content


@app.cls(image=image, secrets=secrets, timeout=600)
class MarketResearcher:
    @modal.method()
    def research(self, idea_json: str) -> str:
        prompt = f\"\"\"
You are the MarketResearchAgent in an autonomous startup studio.

Evaluate this startup idea from a market perspective:

{idea_json}

Return valid JSON with:
target_customer, pain_severity, market_size_score, competition_score, urgency_score,
why_market_exists, key_competitor_type, go_to_market_hint

Use integer scores from 1 to 10.
\"\"\"

        response = _client().chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content


@app.cls(image=image, secrets=secrets, timeout=600)
class FeasibilityAnalyst:
    @modal.method()
    def evaluate(self, idea_json: str) -> str:
        prompt = f\"\"\"
You are the FeasibilityAgent in an autonomous startup studio.

Evaluate this startup idea:

{idea_json}

Return valid JSON with:
technical_feasibility_score, monetization_score, defensibility_score,
execution_complexity_score, overall_score, biggest_risk, why_now, recommendation

Use integer scores from 1 to 10.
\"\"\"

        response = _client().chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content


@app.cls(image=image, secrets=secrets, timeout=600)
class PitchWriter:
    @modal.method()
    def create_pitch(self, idea_json: str, market_report_json: str, feasibility_report_json: str) -> str:
        prompt = f\"\"\"
You are the PitchAgent in an autonomous startup studio.

Startup idea:
{idea_json}

Market report:
{market_report_json}

Feasibility report:
{feasibility_report_json}

Return valid JSON with:
elevator_pitch, investor_angle, launch_wedge, why_it_wins, pitch_score

Use an integer score from 1 to 10.
\"\"\"

        response = _client().chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content
"""

with open("startup_modal_service.py", "w") as f:
    f.write(startup_modal_service)

print("Wrote startup_modal_service.py")

Wrote startup_modal_service.py


## Deploy the Modal service

In [ ]:

# Run modal service after Modal token and secret are set up.
!modal deploy startup_modal_service.py


## Data models

In [13]:
@dataclass
class StartupIdea:
    name: str
    one_liner: str
    problem: str
    solution: str
    target_market: str
    business_model: str
    moat: str
    founder_fit: str = ""


@dataclass
class EvaluatedStartup:
    idea: StartupIdea
    market_score: float
    feasibility_score: float
    pitch_score: float
    overall_score: float
    market_summary: str
    feasibility_summary: str
    pitch: str
    biggest_risk: str
    why_now: str


class MarketResearchReport(BaseModel):
    target_customer: str
    pain_severity: int = Field(ge=1, le=10)
    market_size_score: int = Field(ge=1, le=10)
    competition_score: int = Field(ge=1, le=10)
    urgency_score: int = Field(ge=1, le=10)
    why_market_exists: str
    key_competitor_type: str
    go_to_market_hint: str


class FeasibilityReport(BaseModel):
    technical_feasibility_score: int = Field(ge=1, le=10)
    monetization_score: int = Field(ge=1, le=10)
    defensibility_score: int = Field(ge=1, le=10)
    execution_complexity_score: int = Field(ge=1, le=10)
    overall_score: int = Field(ge=1, le=10)
    biggest_risk: str
    why_now: str
    recommendation: str


class PitchReport(BaseModel):
    elevator_pitch: str
    investor_angle: str
    launch_wedge: str
    why_it_wins: str
    pitch_score: int = Field(ge=1, le=10)

## Memory agent with ChromaDB

In [ ]:

DB_PATH = "startup_studio_db"
COLLECTION_NAME = "startup_ideas"

chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
encoder = SentenceTransformer(EMBEDDING_MODEL)

seed_ideas = [
    {
        "id": "seed_1",
        "name": "TutorFlow",
        "description": "AI study coach that adapts lessons, quizzes, and revision plans for students.",
        "industry": "education",
        "business_model": "subscription"
    },
    {
        "id": "seed_2",
        "name": "CarbonPilot",
        "description": "AI software for SMBs to measure carbon footprint and suggest reduction actions.",
        "industry": "climate",
        "business_model": "B2B SaaS"
    },
    {
        "id": "seed_3",
        "name": "ClinicNote",
        "description": "AI assistant that turns doctor-patient conversations into compliant visit notes.",
        "industry": "healthcare",
        "business_model": "B2B SaaS"
    }
]

existing = set(collection.get().get("ids", []))
new_ids, new_docs, new_meta = [], [], []
for item in seed_ideas:
    if item["id"] not in existing:
        new_ids.append(item["id"])
        new_docs.append(f'{item["name"]}: {item["description"]}')
        new_meta.append({
            "industry": item["industry"],
            "business_model": item["business_model"]
        })

if new_ids:
    embeddings = encoder.encode(new_docs).tolist()
    collection.add(ids=new_ids, documents=new_docs, embeddings=embeddings, metadatas=new_meta)

class MemoryAgent:
    name = "Memory Agent"

    def __init__(self, collection, encoder):
        self.collection = collection
        self.encoder = encoder

    def retrieve_similar(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = self.encoder.encode([query]).tolist()[0]
        result = self.collection.query(query_embeddings=[query_embedding], n_results=n_results)
        return result.get("documents", [[]])[0]

    def save(self, idea: StartupIdea) -> str:
        item_id = f"idea_{abs(hash(idea.name + idea.one_liner))}"
        current_ids = set(self.collection.get().get("ids", []))
        if item_id not in current_ids:
            doc = f"{idea.name}: {idea.one_liner}"
            embedding = self.encoder.encode([doc]).tolist()[0]
            self.collection.add(
                ids=[item_id],
                documents=[doc],
                embeddings=[embedding],
                metadatas=[{
                    "industry": idea.target_market,
                    "business_model": idea.business_model
                }]
            )
        return item_id

memory_agent = MemoryAgent(collection, encoder)
print("Memory size:", collection.count())


## Specialist agents connected to Modal

In [15]:
class IdeaGeneratorAgent:
    name = "Idea Generator Agent"

    def __init__(self, memory_agent: MemoryAgent):
        self.memory_agent = memory_agent
        RemoteCls = modal.Cls.from_name("startup-studio-service", "IdeaGenerator")
        self.remote = RemoteCls()
        self.use_modal = True
        print("IdeaGeneratorAgent connected to Modal")

    def generate(self, theme: str, n_ideas: int = 5) -> List[StartupIdea]:
        memory_context = "\n".join(self.memory_agent.retrieve_similar(theme, n_results=3))
        payload = self.remote.generate.remote(theme, memory_context, n_ideas)
        data = json.loads(payload)
        return [StartupIdea(**idea) for idea in data["ideas"]]


class MarketResearchAgent:
    name = "Market Research Agent"

    def __init__(self):
        RemoteCls = modal.Cls.from_name("startup-studio-service", "MarketResearcher")
        self.remote = RemoteCls()
        self.use_modal = True
        print("MarketResearchAgent connected to Modal")

    def research(self, idea: StartupIdea) -> MarketResearchReport:
        idea_json = json.dumps(asdict(idea), indent=2)
        payload = self.remote.research.remote(idea_json)
        return MarketResearchReport.model_validate_json(payload)


class FeasibilityAgent:
    name = "Feasibility Agent"

    def __init__(self):
        RemoteCls = modal.Cls.from_name("startup-studio-service", "FeasibilityAnalyst")
        self.remote = RemoteCls()
        self.use_modal = True
        print("FeasibilityAgent connected to Modal")

    def evaluate(self, idea: StartupIdea) -> FeasibilityReport:
        idea_json = json.dumps(asdict(idea), indent=2)
        payload = self.remote.evaluate.remote(idea_json)
        return FeasibilityReport.model_validate_json(payload)


class PitchAgent:
    name = "Pitch Agent"

    def __init__(self):
        RemoteCls = modal.Cls.from_name("startup-studio-service", "PitchWriter")
        self.remote = RemoteCls()
        self.use_modal = True
        print("PitchAgent connected to Modal")

    def create_pitch(self, idea: StartupIdea, market_report: MarketResearchReport, feasibility_report: FeasibilityReport) -> PitchReport:
        idea_json = json.dumps(asdict(idea), indent=2)
        market_json = market_report.model_dump_json(indent=2)
        feasibility_json = feasibility_report.model_dump_json(indent=2)
        payload = self.remote.create_pitch.remote(idea_json, market_json, feasibility_json)
        return PitchReport.model_validate_json(payload)


## Planning agent

In [16]:

class PlanningAgent:
    name = "Planning Agent"

    def __init__(self):
        self.memory = memory_agent
        self.idea_generator = IdeaGeneratorAgent(self.memory)
        self.market_researcher = MarketResearchAgent()
        self.feasibility_analyst = FeasibilityAgent()
        self.pitch_writer = PitchAgent()

    def score_startup(self, market: MarketResearchReport, feasibility: FeasibilityReport, pitch: PitchReport) -> float:
        market_component = (
            market.market_size_score +
            market.urgency_score +
            (11 - market.competition_score)
        ) / 3

        overall = (
            0.40 * market_component +
            0.40 * feasibility.overall_score +
            0.20 * pitch.pitch_score
        )
        return round(overall, 2)

    def run(self, theme: str, shortlist_size: int = 3) -> List[EvaluatedStartup]:
        ideas = self.idea_generator.generate(theme, n_ideas=5)

        evaluated = []
        for idea in ideas:
            market_report = self.market_researcher.research(idea)
            feasibility_report = self.feasibility_analyst.evaluate(idea)
            pitch_report = self.pitch_writer.create_pitch(idea, market_report, feasibility_report)

            market_score = round(
                (market_report.market_size_score + market_report.urgency_score + (11 - market_report.competition_score)) / 3,
                2
            )
            overall_score = self.score_startup(market_report, feasibility_report, pitch_report)

            evaluated.append(
                EvaluatedStartup(
                    idea=idea,
                    market_score=market_score,
                    feasibility_score=feasibility_report.overall_score,
                    pitch_score=pitch_report.pitch_score,
                    overall_score=overall_score,
                    market_summary=market_report.why_market_exists,
                    feasibility_summary=feasibility_report.recommendation,
                    pitch=pitch_report.elevator_pitch,
                    biggest_risk=feasibility_report.biggest_risk,
                    why_now=feasibility_report.why_now
                )
            )

        evaluated.sort(key=lambda x: x.overall_score, reverse=True)
        winners = evaluated[:shortlist_size]

        if winners:
            self.memory.save(winners[0].idea)

        return winners

    def present_results(self, winners: List[EvaluatedStartup]) -> str:
        lines = ["TOP STARTUP RECOMMENDATIONS", ""]
        for i, item in enumerate(winners, start=1):
            lines.append(f"{i}. {item.idea.name} — {item.idea.one_liner}")
            lines.append(f"   Target market: {item.idea.target_market}")
            lines.append(f"   Business model: {item.idea.business_model}")
            lines.append(f"   Overall score: {item.overall_score}/10")
            lines.append(f"   Biggest risk: {item.biggest_risk}")
            lines.append(f"   Pitch: {item.pitch}")
            lines.append("")
        return "\n".join(lines)


## Run the system with Gradio

In [ ]:

planner = PlanningAgent()
winners = planner.run("AI tools for climate resilience", shortlist_size=3)
print(planner.present_results(winners))


## Gradio UI

In [ ]:

def generate_startup_report(theme: str):
    planner = PlanningAgent()
    winners = planner.run(theme, shortlist_size=3)
    return json.dumps(
        [
            {
                "name": item.idea.name,
                "one_liner": item.idea.one_liner,
                "target_market": item.idea.target_market,
                "business_model": item.idea.business_model,
                "overall_score": item.overall_score,
                "biggest_risk": item.biggest_risk,
                "pitch": item.pitch,
                "why_now": item.why_now,
            }
            for item in winners
        ],
        indent=2
    )

with gr.Blocks(title="Startup Studio") as ui:
    gr.Markdown("# Autonomous Multi-Agent Startup Studio")
    gr.Markdown("Modal-deployed specialist agents + Chroma memory + autonomous planning.")
    theme = gr.Textbox(label="Startup theme", placeholder="e.g. AI for food waste reduction")
    output = gr.Textbox(label="Recommendations", lines=22)
    button = gr.Button("Generate")
    button.click(generate_startup_report, inputs=theme, outputs=output)

ui.launch()
